In [35]:
from selenium.common.exceptions import InvalidArgumentException, TimeoutException, WebDriverException
from selenium.webdriver.chrome.service import Service
from selenium import webdriver

from bs4 import BeautifulSoup

In [8]:
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("start-maximized")
chrome_options.add_argument('--blink-settings=imagesEnabled=false') 
# chrome_options.add_argument("headless")
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_experimental_option('useAutomationExtension', False)
# chrome_options.add_experimental_option('prefs', {'download.default_directory': self.results_dir})

In [9]:
driver = webdriver.Chrome(options=chrome_options)

In [17]:
lid = '2747150211073176'
URL_LATTES_ID16 = 'http://lattes.cnpq.br/{0}'
url = URL_LATTES_ID16.format(lid)
url

'http://lattes.cnpq.br/2747150211073176'

In [14]:
URL = 'http://buscatextual.cnpq.br/buscatextual/preview.do?metodo=apresentar&id={0}'
url = URL.format('2747150211073176')
url

'http://buscatextual.cnpq.br/buscatextual/preview.do?metodo=apresentar&id=2747150211073176'

In [18]:
driver.get(url)

In [22]:
lid10 = driver.current_url.split('id=')[1]

In [25]:
URL = 'http://buscatextual.cnpq.br/buscatextual/preview.do?metodo=apresentar&id={0}'
url = URL.format(lid10)
driver.get(url)

In [26]:
cmd_open_cv = 'abreCV()'
driver.execute_script(cmd_open_cv)

In [32]:
window = driver.window_handles[1]
driver.switch_to.window(window)

In [33]:
html = driver.page_source

In [34]:
with open('data//html/cv.html', 'w') as f:
    f.write(html)

In [36]:
with open("data/html/cv.html", "r", encoding="latin-1") as f:
    soup = BeautifulSoup(f, "html.parser")

In [37]:
ancora = soup.find("a", {"name": "ProducoesCientificas"})
print(ancora)

<a name="ProducoesCientificas" tabindex="146"><h1 tabindex="0">ProduÃ§Ãµes</h1></a>


In [ ]:
productions = ancora.find_parent("div", class_="title-wrapper")
print(productions)

artigos-completos

In [60]:
from urllib.parse import urlparse, parse_qs, unquote
import request

ModuleNotFoundError: No module named 'request'

In [40]:
div_artigos = productions.find("div", id="artigos-completos")
list_artigos = div_artigos.find_all("div", class_="artigo-completo")
len(list_artigos)

297

In [44]:
artigo = list_artigos[0]
citado = artigo.find("span", class_="citado")
cvuri = citado['cvuri']
print(cvuri)

/buscatextual/servletcitacoes?doi=10.1016/j.fsi.2025.110959&issn=10504648&volume=168&issue=&paginaInicial=110959&titulo=Immunometabolic costs of parasitism under warming: Impaired mitochondrial function and thermal tolerance in an Amazonian fish&sequencial=1&nomePeriodico=FISH & SHELLFISH IMMUNOLOGY


In [47]:
parsed = urlparse(cvuri)
params = parse_qs(parsed.query, keep_blank_values=True)

In [48]:
params

{'doi': ['10.1016/j.fsi.2025.110959'],
 'issn': ['10504648'],
 'volume': ['168'],
 'issue': [''],
 'paginaInicial': ['110959'],
 'titulo': ['Immunometabolic costs of parasitism under warming: Impaired mitochondrial function and thermal tolerance in an Amazonian fish'],
 'sequencial': ['1'],
 'nomePeriodico': ['FISH '],
 ' SHELLFISH IMMUNOLOGY': ['']}

In [54]:
c_doi = []
s_doi = []
for artigo in list_artigos:
    citacao = artigo.find("span", class_="citado")
    if citacao:
        parsed = urlparse(citacao['cvuri'])
        params = parse_qs(parsed.query, keep_blank_values=True)
        [doi] = params['doi'] 
        if doi == '':
            s_doi.append(params)
            print(doi)
        else:
            c_doi.append(params)
            print(doi)


10.1016/j.fsi.2025.110959
10.1111/jfb.70326
10.1007/s10646-025-03022-3
10.1098/rstb.2025.0059
10.1007/s00360-025-01651-y
10.1111/jfb.16050
10.1111/jfb.16063
10.1590/1806-9479.2025.284419
10.1111/jfb.16054
10.1242/jeb.247610
10.1111/jfb.70002
10.1002/ece3.70824
10.1111/jfb.70012
10.1111/jfb.70022
10.11111/jfb.70021
10.1111/jfb.70059
10.14201/reb20231021
10.1111/jfb.70060
10.1590/0001-3765202520241204
10.1590/0001-3765202520250254
10.1016/j.jtemin.2025.100257
10.1111/jfb.70157
10.1111/jfb.70217
10.1007/s11033-025-10991-5
10.1111/jfb.70238
10.1126/science.adr4029
10.1139/cjz-2025-0061
10.1016/j.actatropica.2025.107930
10.1080/03601234.2025.2601943
10.1007/s00360-025-01639-8
10.1016/j.anireprosci.2024.107412
10.21577/0103-5053.20240015
10.1016/j.cbpa.2024.111625
10.1016/j.scitotenv.2024.171379
10.1007/s00360-024-01552-6
10.1590/1982-0224-2023-0114
10.1016/j.scitotenv.2024.174173
10.1016/j.actatropica.2024.107328
10.1007/s10499-024-01637-7
10.1371/journal.pone.0306985
10.1242/jeb.247255
10.

In [64]:
doi = c_doi[0]['doi'][0]
doi

'10.1016/j.fsi.2025.110959'

In [77]:
import httpx
import json

In [67]:
url = f"https://api.crossref.org/v1/works/{doi}"
r = httpx.get(url)
r.status_code

200

In [78]:
error = []
with open("data/artigos/val.jsonl", "w", encoding="utf-8") as f:
    
    for i in c_doi:
        doi = i['doi'][0]
        url = f"https://api.crossref.org/v1/works/{doi}"
        r = httpx.get(url)
        print(r.status_code)
        if r.status_code == 200:
            item = r.json()['message']
            json.dump(item, f)
            f.write("\n")
        else:
            print(f"Error fetching data for DOI: {doi}, status code: {r.status_code}")
            error.append(i)
            



200
200
200
200
200
200
200
200
200
200
200
200
200
200
404
Error fetching data for DOI: 10.11111/jfb.70021, status code: 404
200
404
Error fetching data for DOI: 10.14201/reb20231021, status code: 404
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
404
Error fetching data for DOI: 10.1016 / j.scitotenv.2020.138628, status code: 404
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
2

In [73]:
item.keys()

dict_keys(['indexed', 'reference-count', 'publisher', 'license', 'funder', 'content-domain', 'short-container-title', 'published-print', 'DOI', 'type', 'created', 'page', 'update-policy', 'source', 'is-referenced-by-count', 'special_numbering', 'title', 'prefix', 'volume', 'author', 'member', 'reference', 'container-title', 'original-title', 'language', 'link', 'deposited', 'score', 'resource', 'subtitle', 'short-title', 'issued', 'references-count', 'alternative-id', 'URL', 'relation', 'ISSN', 'issn-type', 'subject', 'published', 'assertion', 'article-number'])

In [75]:
for k, v in item.items():
    print(k,"--", v)

indexed -- {'date-parts': [[2026, 3, 12]], 'date-time': '2026-03-12T14:03:23Z', 'timestamp': 1773324203523, 'version': '3.50.1'}
reference-count -- 63
publisher -- Elsevier BV
license -- [{'start': {'date-parts': [[2026, 1, 1]], 'date-time': '2026-01-01T00:00:00Z', 'timestamp': 1767225600000}, 'content-version': 'tdm', 'delay-in-days': 0, 'URL': 'https://www.elsevier.com/tdm/userlicense/1.0/'}, {'start': {'date-parts': [[2026, 1, 1]], 'date-time': '2026-01-01T00:00:00Z', 'timestamp': 1767225600000}, 'content-version': 'tdm', 'delay-in-days': 0, 'URL': 'https://www.elsevier.com/legal/tdmrep-license'}, {'start': {'date-parts': [[2026, 1, 1]], 'date-time': '2026-01-01T00:00:00Z', 'timestamp': 1767225600000}, 'content-version': 'stm-asf', 'delay-in-days': 0, 'URL': 'https://doi.org/10.15223/policy-017'}, {'start': {'date-parts': [[2026, 1, 1]], 'date-time': '2026-01-01T00:00:00Z', 'timestamp': 1767225600000}, 'content-version': 'stm-asf', 'delay-in-days': 0, 'URL': 'https://doi.org/10.1522